In [34]:
from dotenv import load_dotenv
import os
import os
from smolagents import OpenAIServerModel
from smolagents import DuckDuckGoSearchTool, CodeAgent, ToolCallingAgent, VisitWebpageTool
from sec_api import QueryApi
import pandas as pd
from langchain.docstore.document import Document
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from smolagents import Tool

load_dotenv()
SEC_API_KEY = os.getenv("SEC_API_KEY")
LLM_API_KEY = os.getenv("LLM_API_KEY")

In [26]:
news_data_path = "./Data/news-2007-2023.csv"

In [46]:
new_df = pd.read_csv(news_data_path)
new_df.to_dict(orient='records')

[{'Title': 'TSX Slightly Down, Books Weekly Gains',
  'Tag': 'Stock Market',
  'Content': 'TSX Slightly Down, Books Weekly GainsUnited States\xa0Stock MarketThe S&P/TSX Composite index ended marginally in the red at the 20,260 level on Friday, the highest since late May, snapping the sharp gains from the prior three sessions while adding 2.2% on the week amid continuous hopes that lower inflation will ease central banks’ prolonged tightening pressure. Energy producers led the losses of the session and slipped 2.3%, tracking lower oil prices. Also, policy-sensitive tech shares edged down by 0.2%, tracking their peers on Nasdaq. On the other hand, the heavyweight financial sector gained 0.3% following some upbeat corporate results for US banking giants. Among stocks, TELUS International tumbled 30.5% after the IT services company estimated a loss for the second quarter results. Pine Cliff Energy fell 6.9% after being downgraded by Stifel. Also, First Quantum Minerals slipped 0.7% after R

In [52]:
def create_documents_from_excel(df: pd.DataFrame):
    documents = []
    df_dict = df.to_dict(orient='records')
    for item in df_dict:
        string_item = ""
        for key, value in item.items():
            string_item = string_item + key + "\n" + value + "\n"
            document = Document(page_content=string_item)
            documents.append(document)
    return documents

In [53]:
documents = create_documents_from_excel(new_df)

In [ ]:
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [60]:
vectorStore = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

In [62]:
vectorStore.save_local("finance-news")

In [6]:
class RetrieverTool(Tool):
    name = "retriever"
    description = "Uses semantic search to retrieve the parts of transformers documentation that could be most relevant to answer your query."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be semantically close to your target documents. Use the affirmative form rather than a question.",
        }
    }
    output_type = "string"

    def __init__(self, vector_store_path, **kwargs):
        super().__init__(**kwargs)
        # Initialize the retriever with our processed documents
        self.embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
        self.vector_store = FAISS.load_local(vector_store_path, self.embedding, allow_dangerous_deserialization=True)

    def forward(self, query: str) -> str:
        """Execute the retrieval based on the provided query."""
        assert isinstance(query, str), "Your search query must be a string"

        # Retrieve relevant documents
        # faiss_retriever = self.vector_store.as_retriever(search_kwargs = {"k": 5})
        # ensemble_retriever = EnsembleRetriever(
        #     retrievers=[faiss_retriever], weights=[1]
        # )
        
        # docs = ensemble_retriever.get_relevant_documents(query)
        docs = self.vector_store.similarity_search(
            query=query,
            k= 5
        )
        
        # Format the retrieved documents for readability
        return "\nRetrieved documents:\n" + "".join(
            [
                f"\n\n===== Document {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

In [70]:
tool = RetrieverTool("./finance-news")

In [75]:
tool.forward("AT&T Hits All-time LowUnited States stocksAT&T decreased to an all-time low of 14.393 days ago")

'\nRetrieved documents:\n\n\n===== Document 0 =====\nTitle\nAT&T Hits All-time Low\nTag\nstocks\nContent\nAT&T Hits All-time LowUnited States\xa0stocksAT&T decreased to an all-time low of 14.393 days ago\n\n\n===== Document 1 =====\nTitle\nAT&T Hits 14-week Low\nTag\nstocks\nContent\nAT&T Hits 14-week LowUnited States\xa0stocksAT&T decreased to a 14-week low of 16.942023-05-04T13:47:18.833\n\n\n===== Document 2 =====\nTitle\nAT&T Hits 16-week Low\nTag\nstocks\nContent\nAT&T Hits 16-week LowUnited States\xa0stocksAT&T decreased to a 16-week low of 16.52023-05-18T15:08:57.42\n\n\n===== Document 3 =====\nTitle\nAT&T Hits 33-week Low\nTag\nstocks\nContent\nAT&T Hits 33-week LowUnited States\xa0stocksAT&T decreased to a 33-week low of 14.982023-06-02T14:27:56.7\n\n\n===== Document 4 =====\nTitle\nAT&T Hits 4-week Low\nTag\nstocks\nContent\nAT&T Hits 4-week LowUnited States\xa0stocksAT&T decreased to a 4-week low of 18.262023-04-20T13:34:13.49\n'

In [35]:

class SECTool(Tool):
    name = "SECFilingSearcher" # A slightly more descriptive name
    description = (
        "Searches for SEC filings (10-Q or 8-K) for a given company ticker "
        "within a specified date range. Returns a dictionary containing the filings."
    )
    inputs = {
        "ticker": {
            "type": "string",
            "description": "The company ticker symbol (e.g., AAPL, MSFT).",
        },
        "from_date": {
            "type": "string",
            "description": "The start date for the search, in YYYY-MM-DD format (e.g., '2023-01-01').",
        },
        "to_date": {
            "type": "string",
            "description": "The end date for the search, in YYYY-MM-DD format (e.g., '2023-03-31').",
        },
        "form_type": {
            "type": "string",
            # If smolagents supports enum type in schema, that would be better.
            # Otherwise, a clear description is key.
            "description": "The type of SEC form to search for. Must be either '10-Q' or '8-K'.",
        }
    }
    output_type = "string" # Output text from passing url from queryApi.get_filings()

    def __init__(self, sec_api_key: str, **kwargs):
        super().__init__(**kwargs)
        if not sec_api_key:
            raise ValueError("SEC API key must be provided to SECTool.")
        self.queryApi = QueryApi(api_key=sec_api_key)

    def forward(self, ticker: str, from_date: str, to_date: str, form_type: str) -> dict:
        """
        Fetches SEC filings based on the provided criteria.

        Parameters:
         - ticker: string, the company ticker symbol.
         - from_date: string, the starting date for SEC search (YYYY-MM-DD).
         - to_date: string, the ending date for SEC search (YYYY-MM-DD).
         - form_type: string, the SEC form type, must be '10-Q' or '8-K'.
        """
        # Basic validation for form_type
        if form_type not in ["10-Q", "8-K", "10-K"]:
            return {
                "error": f"Invalid form_type: '{form_type}'. Must be '10-Q' or '8-K'."
            }
        
        # You might want to add date format validation here if needed,
        # though the API might handle malformed dates with an error.

        query_string = f"ticker:{ticker} AND filedAt:[{from_date} TO {to_date}] AND formType:\"{form_type}\""
        query = {
            "query": query_string,
            "from": "0",  # Start from the first result
            "size": "10", # Number of results to return
            "sort": [{ "filedAt": { "order": "desc" } }]
        }

        try:
            print(f"Executing SEC API query: {query}") # For debugging
            filings = self.queryApi.get_filings(query)
            # The 'filings' variable here is expected to be a dictionary,
            # which might contain a list of actual filing documents under a specific key.
            # For example: {"total": {"value": 5, "relation": "eq"}, "filings": [...], "query": ...}
            # This entire dictionary is returned as per output_type = "dict".
            return filings
        except Exception as e:
            # Log the exception for server-side debugging
            print(f"Error calling SEC API: {e}")
            # Return a dictionary with an error message for the agent
            return {
                "error": f"An error occurred while fetching SEC filings for {ticker}: {str(e)}",
                "query_attempted": query_string
            }

In [ ]:

class DataScraperTool(Tool):
    name = "DataScraperTool" # A slightly more descriptive name
    description = (
        "Uses beautifulsoup to extract data from a give web url"
    )
    inputs = {
        "url": {
            "type": "string",
            "description": "Url of the web page",
        },
    }
    output_type = "string" # Output text from passing url from queryApi.get_filings()

    def __init__(self, user_email: str, **kwargs):
        super().__init__(**kwargs)
        if not user_email.endswith("@gmail.com"):
            raise ValueError("Please provdie a valid gmail")
        self.user_email = user_email

    def forward(self, url: str) -> str:
        """Scrape the data from the web page using beautiful soup and python.

        Args:
            url (str): A SEC url of a filing

        Returns:
            str: Return the data scraped from the web page 
        """
        
        import requests
        from bs4 import BeautifulSoup
        
        # Fetch the page
        headers = {'User-Agent': f'Data Extraction Script ({self.user_email})'}
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an exception for HTTP errors

        # Parse with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        # Example 1: Extract all text in the <body>
        body_text = soup.body.get_text(separator='\n', strip=True)
        print("Full document text from <body>:\n")
        print(body_text)

        # (Optional) Example 2: Extract all <table> contents (common in SEC filings)
        tables = soup.find_all('table')
        for idx, table in enumerate(tables, 1):
            print(f"\n---- Table {idx} ----\n")
            print(table.get_text(separator='\n', strip=True))
        

In [36]:
model = OpenAIServerModel(
    model_id="gpt-4.1",
    api_key=LLM_API_KEY,
)
retriever_agent = ToolCallingAgent(tools=[RetrieverTool("./finance-news")], model=model, name="retriever_agent", description = "Retrieve financial news documents from a FAISS vectorstore")
web_search_agent = ToolCallingAgent(tools=[DuckDuckGoSearchTool()], model = model, name = "web_seach_agent", description = "A web based search agent that can look any generic query on internet")


In [38]:
sec_agent = CodeAgent(tools=[SECTool(sec_api_key=os.getenv("SEC_API_KEY")), VisitWebpageTool()], model=model,managed_agents = [],  name="sec_agent", description = "Uses sec api to find the filing information of companies for 8-K and 10-Q filings")


In [42]:
sec_agent.run("What are the key geopolitical risks mentioned in Microsoft's most recent 10-K?")

╭────────────────────────────────────────────── New run - sec_agent ──────────────────────────────────────────────╮
│                                                                                                                 │
│ What are the key geopolitical risks mentioned in Microsoft's most recent 10-K?                                  │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4.1 ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  from datetime import datetime                                                                                    
                                                                                                                   
  # Today's date for latest search window                                                                          
  today = datetime.today().strftime('%Y-%m-%d')                                                                    
  # Look back 1.5 years to be sure to include the latest filing                                                    
  start_date = (datetime.today().replace(year=datetime.today().year - 2)).strftime('%Y-%m-%d')                     
                                                                                                                   
  # Search for 10-K filings for Microsoft                                                                          
  msft_10ks = SECFilingSearcher(                                                                                   
      ticker="MSFT",                                                                                               
      from_date=start_date,                                                                                        
      to_date=today,                                                                                               
      form_type="10-K"                                                                                             
  )                                                                                                                
  print(msft_10ks)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Executing SEC API query: {'query': 'ticker:MSFT AND filedAt:[2023-05-30 TO 2025-05-30] AND formType:"10-K"', 'from': '0', 'size': '10', 'sort': [{'filedAt': {'order': 'desc'}}]}


Execution logs:
{'total': {'value': 2, 'relation': 'eq'}, 'query': {'from': 0, 'size': 10}, 'filings': [{'ticker': 'MSFT', 
'formType': '10-K', 'accessionNo': '0000950170-24-087843', 'cik': '789019', 'companyNameLong': 'MICROSOFT CORP 
(Filer)', 'companyName': 'MICROSOFT CORP', 'linkToFilingDetails': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-20240630.htm', 'description': 'Form 10-K - 
Annual report [Section 13 and 15(d), not S-K Item 405]', 'linkToTxt': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/0000950170-24-087843.txt', 'filedAt': 
'2024-07-30T16:06:22-04:00', 'documentFormatFiles': [{'sequence': '1', 'size': '6860911', 'documentUrl': 
'https://www.sec.gov/ix?doc=/Archives/edgar/data/789019/000095017024087843/msft-20240630.htm', 'description': 
'10-K', 'type': '10-K'}, {'sequence': '2', 'size': '96978', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex4_26.htm', 'description': 'EX-4.26', 
'type': 'EX-4.26'}, {'sequence': '3', 'size': '268169', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex10_5.htm', 'description': 'EX-10.5', 
'type': 'EX-10.5'}, {'sequence': '4', 'size': '35474', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex19_1.htm', 'description': 'EX-19.1', 
'type': 'EX-19.1'}, {'sequence': '5', 'size': '44401', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex19_2.htm', 'description': 'EX-19.2', 
'type': 'EX-19.2'}, {'sequence': '6', 'size': '55143', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex19_3.htm', 'description': 'EX-19.3', 
'type': 'EX-19.3'}, {'sequence': '7', 'size': '13325', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex21.htm', 'description': 'EX-21', 'type': 
'EX-21'}, {'sequence': '8', 'size': '4773', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex23_1.htm', 'description': 'EX-23.1', 
'type': 'EX-23.1'}, {'sequence': '9', 'size': '11145', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex31_1.htm', 'description': 'EX-31.1', 
'type': 'EX-31.1'}, {'sequence': '10', 'size': '11369', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex31_2.htm', 'description': 'EX-31.2', 
'type': 'EX-31.2'}, {'sequence': '11', 'size': '7753', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex32_1.htm', 'description': 'EX-32.1', 
'type': 'EX-32.1'}, {'sequence': '12', 'size': '7375', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex32_2.htm', 'description': 'EX-32.2', 
'type': 'EX-32.2'}, {'sequence': '13', 'size': '46526', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-ex97_1.htm', 'description': 'EX-97.1', 
'type': 'EX-97.1'}, {'sequence': '\xa0', 'size': '30375960', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/0000950170-24-087843.txt', 'description': 
'Complete submission text file', 'type': '\xa0'}], 'periodOfReport': '2024-06-30', 'entities': [{'fiscalYearEnd': 
'0630', 'stateOfIncorporation': 'WA', 'act': '34', 'cik': '789019', 'fileNo': '001-37845', 'irsNo': '911144442', 
'companyName': 'MICROSOFT CORP (Filer)', 'type': '10-K', 'sic': '7372 Services-Prepackaged Software', 'filmNo': 
'241157360', 'undefined': '06 Technology)'}], 'id': 'b80c727c9b0c7ef7fb3c6f15b3f6f3d1', 
'seriesAndClassesContractsInformation': [], 'linkToHtml': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/0000950170-24-087843-index.htm', 'linkToXbrl': 
'', 'dataFiles': [{'sequence': '14', 'size': '2099152', 'documentUrl': 
'https://www.sec.gov/Archives/edgar/data/789019/000095017024087843/msft-20240630.xsd', 'description': 'XBRL 
TAXONOMY EXTENSIO

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  summary = """                                                                                                    
  Unable to access the exact text of Microsoft's most recent 10-K due to access restrictions, but based on         
  industry norms and Microsoft's previous filings, the key geopolitical risks typically mentioned include:         
                                                                                                                   
  1. International trade tensions, especially between the US and China, which can affect Microsoft's ability to    
  sell, license, or supply products and services globally.                                                         
  2. Risks from export controls and sanctions imposed by governments, which may restrict the company's technology  
  transfer or limit operations in certain geographies (e.g., Russia, China, other sanctioned countries).           
  3. Changes in local or regional laws, regulations, or government policies, including those affecting antitrust,  
  privacy, and data localization, particularly in the European Union and China.                                    
  4. Cybersecurity risks associated with nation-state actors targeting digital infrastructure, products, or        
  customer data.                                                                                                   
  5. Political instability or armed conflict in regions where Microsoft operates, which can disrupt supply         
  chains, partners, or customers.                                                                                  
                                                                                                                   
  For the precise wording and latest updates, please consult the Risk Factors section (Item 1A) of the 2024        
  Microsoft 10-K filing on the SEC EDGAR website once access is available.                                         
  """                                                                                                              
  final_answer(summary)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 
Unable to access the exact text of Microsoft's most recent 10-K due to access restrictions, but based on industry 
norms and Microsoft's previous filings, the key geopolitical risks typically mentioned include:

1. International trade tensions, especially between the US and China, which can affect Microsoft's ability to sell,
license, or supply products and services globally.
2. Risks from export controls and sanctions imposed by governments, which may restrict the company's technology 
transfer or limit operations in certain geographies (e.g., Russia, China, other sanctioned countries).
3. Changes in local or regional laws, regulations, or government policies, including those affecting antitrust, 
privacy, and data localization, particularly in the European Union and China.
4. Cybersecurity risks associated with nation-state actors targeting digital infrastructure, products, or customer 
data.
5. Political instability or armed conflict in regions where Microsoft operates, which can disrupt supply chains, 
partners, or customers.

For the precise wording and latest updates, please consult the Risk Factors section (Item 1A) of the 2024 Microsoft
10-K filing on the SEC EDGAR website once access is available.

[Step 4: Duration 8.00 seconds| Input tokens: 19,764 | Output tokens: 1,005]

"\nUnable to access the exact text of Microsoft's most recent 10-K due to access restrictions, but based on industry norms and Microsoft's previous filings, the key geopolitical risks typically mentioned include:\n\n1. International trade tensions, especially between the US and China, which can affect Microsoft's ability to sell, license, or supply products and services globally.\n2. Risks from export controls and sanctions imposed by governments, which may restrict the company's technology transfer or limit operations in certain geographies (e.g., Russia, China, other sanctioned countries).\n3. Changes in local or regional laws, regulations, or government policies, including those affecting antitrust, privacy, and data localization, particularly in the European Union and China.\n4. Cybersecurity risks associated with nation-state actors targeting digital infrastructure, products, or customer data.\n5. Political instability or armed conflict in regions where Microsoft operates, which c

In [99]:
manager_agent = CodeAgent(
    tools = [],
    model=model,
    managed_agents = [retriever_agent, web_search_agent]
)

In [105]:
manager_agent.run("microsoft hits 4-weeks high")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ microsoft hits 4-weeks high                                                                                     │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4.1 ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = retriever_agent("Microsoft hits 4-weeks high")                                                          
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭─────────────────────────────────────────── New run - retriever_agent ───────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'retriever_agent'.                                                                 │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Microsoft hits 4-weeks high                                                                                     │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ OpenAIServerModel - gpt-4.1 ───────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Microsoft has recently hit a 4-week high, with its stock most recently reaching $348. This         
  reflects strong performance and investor interest, showing a positive trend compared to previous 4-week highs    
  such as $266.52.")                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: Microsoft has recently hit a 4-week high, with its stock most recently reaching $348. This 
reflects strong performance and investor interest, showing a positive trend compared to previous 4-week highs such 
as $266.52.

[Step 2: Duration 2.91 seconds| Input tokens: 4,660 | Output tokens: 191]

'Microsoft has recently hit a 4-week high, with its stock most recently reaching $348. This reflects strong performance and investor interest, showing a positive trend compared to previous 4-week highs such as $266.52.'

In [31]:
import requests
from bs4 import BeautifulSoup

url = 'https://www.sec.gov/ix?doc=/Archives/edgar/data/789019/000095017023035122/msft-20230630.htm'


In [33]:

# Fetch the page
headers = {'User-Agent': 'Data Extraction Script (khushalgoyal77@gmail.com)'}
response = requests.get(url, headers=headers)
response.raise_for_status()  # Raise an exception for HTTP errors

# Parse with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Example 1: Extract all text in the <body>
body_text = soup.body.get_text(separator='\n', strip=True)
print("Full document text from <body>:\n")
print(body_text)

# (Optional) Example 2: Extract all <table> contents (common in SEC filings)
tables = soup.find_all('table')
for idx, table in enumerate(tables, 1):
    print(f"\n---- Table {idx} ----\n")
    print(table.get_text(separator='\n', strip=True))

Full document text from <body>:

Please enable JavaScript to use the EDGAR Inline XBRL Viewer.
